# Colas desde reporting
Generador independiente de `archivados/bases_plantillas/archivos/reporte_colas.xlsx`, con la hoja **Datos** y la tabla **t_datos**. Conserva la consulta Genesys y sus 40 columnas.

`DESDE = HASTA = None` mantiene el alcance original, sin filtro de fechas. Para acotarlo, usar `date(año, mes, día)`; ambos extremos son inclusivos.

Las duraciones conservan las **horas decimales** entregadas por la vista, sin dividir por 24. Los tiempos terminados en `_seg` siguen en segundos. `intervalo` es una hora de reloj de Excel. Los identificadores son texto y los nulos quedan vacíos.

La extracción y los formatos se comparten con los scripts finales mediante `src/analytics/datos_reportes.py`. El Excel se valida contra los datos antes de sustituir el archivo anterior.

In [1]:
from datetime import date
from pathlib import Path
import polars as pl
from IPython.display import display
from analytics.excel_reportes import TablaDatos, generar_libro

actual = Path.cwd().resolve()
BASE_DIR = next((p for p in (actual, *actual.parents)
                 if (p / "db_conn.py").is_file() and (p / "pyproject.toml").is_file()), None)
if BASE_DIR is None:
    raise FileNotFoundError("Abrir el notebook dentro del proyecto.")
import sys
if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

from analytics.datos_reportes import cargar_colas, formatos_colas
DESDE = None
HASTA = None
RUTA_EXCEL = BASE_DIR / "archivados" / "bases_plantillas" / "archivos" / "reporte_colas.xlsx"

In [2]:
print("Consultando Colas...")
df = cargar_colas(DESDE, HASTA, BASE_DIR)
print(f"Colas: {df.height:,} filas; {df.width} columnas.")
display(pl.DataFrame({"columna": list(df.schema), "tipo": [str(t) for t in df.schema.values()]}))

Consultando Colas...


Colas: 258,525 filas; 40 columnas.


columna,tipo
str,str
"""fecha""","""Date"""
"""intervalo""","""Time"""
"""tipo_medios""","""String"""
"""id_cola""","""String"""
"""nombre_cola""","""String"""
…,…
"""tipo_campana""","""String"""
"""cliente""","""String"""
"""unidad_negocio""","""String"""


In [3]:
print("Escribiendo y validando el Excel...")
generar_libro([TablaDatos("t_datos", "Datos", df, formatos_colas())], RUTA_EXCEL)
print(f"Generado: {RUTA_EXCEL.name}; {df.height:,} filas.")

Escribiendo y validando el Excel...


Generado: reporte_colas.xlsx; 258,525 filas.
